# SIMT 抽象硬件架构

## 概述

本节我们学习 SIMT 编程相关的抽象硬件架构。我们编写的 SIMT 算子代码最终要在 NPU 硬件上执行，在编写算子前先了解硬件架构，有助于我们充分利用硬件资源，从而写出更高性能的 SIMT 算子。

为了降低初学者的理解门槛，本节不直接讲解昇腾 AI 处理器复杂的原生硬件，而是介绍 Ascend C SIMT 对并行计算硬件做的一层轻量化抽象。掌握这层抽象模型，就足以理解后续算子开发中的数据流动与资源配置。

### 前置要求

- 已学习 03.02 编程模型概述，了解 Ascend C 算子运行在 host 侧和 device 侧的基本关系。
- 具备基础 C/C++ 概念，能够理解内存和并行计算中的线程概念。
- 本小节为理论讲解，不依赖在线硬件环境。

### 学习目标

学完本小节后，你应该能够：

- 描述 AI 处理器、Vector Core、Unified Buffer、Global Memory 之间的关系。
- 理解寄存器、共享内存（Shared Memory）和 Data Cache 在多线程计算中的作用。
- 说清楚一个线程从 Global Memory 读取数据到寄存器，数据大致会经过哪些硬件资源。
- 理解这套硬件架构如何支撑 SIMT 的大规模线程并行。

### 小节内容

- 从单个线程的计算流程说起
- SIMT 的硬件构成
- 数据如何在层级间流动
- 线程之间如何共享存储资源

### 从单个线程的计算流程说起

SIMT 通过大量线程并发执行来实现计算加速。在理解多线程的硬件架构之前，我们先回顾**单个线程**的通用计算流程：

待计算的数据先被加载到内存中，再从内存通过多级缓存逐级加载到寄存器上；计算单元在寄存器上完成计算后，结果再从寄存器逐级写回内存。

![通用计算流程](images/memory_compute_flow.svg)

这里有一个关键认识：**计算单元只能直接读写寄存器**，无法直接对内存中的数据做运算。因此数据必须先「搬」到离计算单元最近的寄存器，算完再「搬」回内存。多级缓存的作用，就是让这条搬运路径更高效。

### SIMT 的硬件构成

单个线程的流程理解之后，再看多线程。为了让大量线程并发执行，昇腾 AI 处理器内部设计了多个计算核心，每个计算核心包含大量计算单元，能够并发执行大量线程（1K 以上）。SIMT 抽象硬件架构如下图所示。

![SIMT抽象硬件架构教学图](images/SIMT抽象硬件架构图.png)

图中各组件的关系可以这样理解：

- **Vector Core** 是 SIMT 编程的计算核心。每个 AI 处理器内部有多个 Vector Core。
- 每个 **Vector Core** 内部包含多个**计算单元**、一块 **Unified Buffer** 和大量**寄存器**。其中 Unified Buffer 又被划分为 **Shared Memory** 和 **Data Cache** 两部分。
- L2 Cache 是Level 2缓存，被所有 Vector Core 共享。
- **Global Memory** 是核外的全局内存空间，被所有 Vector Core 共享。

### 数据如何在层级间流动

回到数据流动。Global Memory 承载着所有算子的输入输出数据。每个线程计算时，把数据从 Global Memory 通过多级缓存逐级加载到寄存器中，在寄存器中完成计算后，再逐级写回 Global Memory。

![数据加载与写回路径](images/memory_compute_flow_simt.svg) 

### 线程之间如何共享存储资源

一个 Vector Core 中可以并发执行大量线程。这些线程在存储资源上有「独享」和「共享」之分：

- **独享**：每个线程拥有独立的寄存器资源，用于保存自己的局部数据，互不干扰。
- **共享**：同一个 Vector Core 内的线程共享 Shared Memory 和 Data Cache。其中 Shared Memory 通常用于 Vector Core 内线程间交换数据；Data Cache 用于缓存从 Global Memory 加载的数据，以提高 Global Memory 的访存效率。

开发者可以根据算子的实际需求，配置 Unified Buffer 分配给 Shared Memory 和 Data Cache 的比例。具体划分方法将在后续的《内存层级》章节中讲解。

### 术语速查

| 术语 | 说明 |
| --- | --- |
| SIMT | 单指令多线程，大量线程执行同一类指令，但各自处理不同的数据。 |
| AI 处理器 | 承载 SIMT 计算任务的整体硬件平台，内部包含多个 Vector Core。 |
| Vector Core | AI 处理器内部的计算核心，包含计算单元和片上存储等资源。 |
| 计算单元 | 完成具体算术或逻辑计算，只能直接读写寄存器。 |
| Unified Buffer | Vector Core 内部的片上缓存资源，可被划分为 Shared Memory 和 Data Cache。 |
| Shared Memory | Unified Buffer 的一部分，供 Vector Core 内线程共享，常用于线程间交换数据。 |
| Data Cache | Unified Buffer 的一部分，用于缓存从 Global Memory 加载的数据。 |
| Global Memory | 核外全局内存，容量最大，被所有 Vector Core 共享。 |
| Register（寄存器） | 每个线程独立使用的高速局部存储资源，离计算单元最近。 |

## 小节小结

本小节从硬件视角建立了 SIMT 编程的基本地图：

- **层级关系**：AI 处理器内部包含多个 Vector Core；每个 Vector Core 内部包含计算单元、Unified Buffer（划分为 Shared Memory 和 Data Cache）和寄存器；核外的 Global Memory 作为全局内存，被所有 Vector Core 共享。
- **数据流动**：计算单元只能直接读写寄存器，所以数据要从 Global Memory 逐级加载到寄存器、算完再逐级写回 Global Memory。
- **资源划分**：同一 Vector Core 内的线程独享各自的寄存器，但共享 Shared Memory 和 Data Cache。

对开发者来说，后续写 SIMT 算子时需要持续关注两类问题：一类是数据如何在 Global Memory、Data Cache、Shared Memory 和寄存器之间流动，另一类是如何配置硬件资源以实现更高的性能。这些都建立在本节这套抽象硬件模型之上。

至此我们已经知道：昇腾 AI 处理器中有多个 Vector Core，每个 Vector Core 都能并发执行大量线程。但这些线程具体如何分工、协同完成大量数据的计算，将在下一章节《线程架构》中讲解。

## 课后练习

本节介绍了 SIMT 抽象硬件架构中 AI 处理器、Vector Core、Unified Buffer、Global Memory 以及寄存器等存储资源的关系，请根据学习内容完成以下题目进行自测。

1. （判断题）Global Memory 位于 Vector Core 内部，每个 Vector Core 独占一份，互不共享。

2. （单选题）SIMT 抽象硬件架构中，Unified Buffer 被划分为哪两个部分？  
    A. L2 Cache 和 Data Cache  
    B. Shared Memory 和 Data Cache  
    C. 寄存器和栈空间  
    D. Global Memory 和 Shared Memory  

3. （单选题）一个线程计算数据时，数据从 Global Memory 加载到寄存器、计算后再写回 Global Memory，完整路径是下列哪一项？  
    A. Global Memory → 寄存器 → Global Memory  
    B. Global Memory → Shared Memory → 寄存器 → Shared Memory → Global Memory  
    C. Global Memory → L2 Cache → Data Cache → Register → Data Cache → L2 Cache → Global Memory  
    D. Global Memory → Data Cache → L2 Cache → Register → L2 Cache → Data Cache → Global Memory  

4. （多选题）以下关于 SIMT 抽象硬件架构的说法，哪些是正确的？  
    A. 每个 AI 处理器内部包含多个 Vector Core  
    B. Shared Memory 通常用于 Vector Core 内线程间交换数据  
    C. Global Memory 被所有 Vector Core 共享  
    D. 同一个 Vector Core 内的所有线程共享同一组寄存器  

**执行以下代码获取答案。**

## 参考答案

运行下面的单元格查看课后练习参考答案。


In [ ]:
!cat answer/03.04.01_answer.txt
